##### Copyright 2026 Google LLC.

In [2]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemini API: File API Quickstart

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/File_API.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

The Gemini API supports prompting with text, image, and audio data, also known as *multimodal* prompting. You can include text, image,
and audio in your prompts. For small images, you can point the Gemini model
directly to a local file when providing a prompt. For larger text files, images, videos, and audio, upload the files with the [File
API](https://ai.google.dev/api/rest/v1beta/files) before including them in
prompts.

The File API lets you store up to 20GB of files per project, with each file not
exceeding 2GB in size. Files are stored for 48 hours and can be accessed with
your API key for generation within that time period. It is available at no cost in all regions where the [Gemini API is
available](https://ai.google.dev/available_regions).

For information on valid file formats (MIME types) and supported models, see the documentation on
[supported file formats](https://ai.google.dev/tutorials/prompting_with_media#supported_file_formats)
and view the text examples at the end of this guide.

This guide shows how to use the File API to upload a media file and include it in a `GenerateContent` call to the Gemini API. For more information, see the [code
samples](../quickstarts/file-api).


> **Note:** This notebook uses the [Interactions API](https://ai.google.dev/gemini-api/docs/interactions), the latest way to interact with Gemini models. Looking for the `generateContent` version? Check the [archive branch](https://github.com/google-gemini/cookbook/blob/archive/generate-content-api/quickstarts/File_API.ipynb).

### Install dependencies

In [ ]:
%pip install -U -q "google-genai>=2.9.0"  # 1.57 for File API features # 2.0 is needed to use the interactions API

### Authentication

**Important:** The File API uses API keys for authentication and access. Uploaded files are associated with the API key's cloud project. Your API key must be stored in a Colab Secret named `GEMINI_API_KEY`.

#### Set up your API key

To run the following cell, your API key must be stored in a Colab Secret named `GEMINI_API_KEY`. If you don't already have an API key, or you're not sure how to create a Colab Secret, see [Authentication ![image](https://storage.googleapis.com/generativeai-downloads/images/colab_icon16.png)](../quickstarts/Authentication.ipynb) for a walkthrough.

In [10]:
from google import genai
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=GEMINI_API_KEY)

## Upload file

The File API lets you upload a variety of multimodal MIME types, including images and audio formats. The File API handles inputs that can be used to generate content with `client.interactions.create` (or `client.models.generate_content`).

The File API accepts files under 2GB in size and can store up to 20GB of files per project. Files last for 2 days and cannot be downloaded from the API.

First, you will prepare a sample image to upload to the API.

Note: You can also [upload your own files](../examples/Upload_files_to_Colab.ipynb) to use.

In [13]:
from IPython.display import Image, display
import urllib.request

urllib.request.urlretrieve(
    "https://storage.googleapis.com/generativeai-downloads/images/jetpack.jpg",
    "image.jpg"
)
display(Image(filename="image.jpg"))

<IPython.core.display.Image object>


Next, you will upload that file to the File API.

In [15]:
sample_file = client.files.upload(file="image.jpg")

print(f"Uploaded file '{sample_file.name}' as: {sample_file.uri}")

Uploaded file 'files/zyg60km4wauf' as: https://generativelanguage.googleapis.com/v1beta/files/zyg60km4wauf


The `response` shows that the File API stored the specified `display_name` for the uploaded file and a `uri` to reference the file in Gemini API calls. Use `response` to track how uploaded files are mapped to URIs.

Depending on your use cases, you could store the URIs in structures such as a `dict` or a database.

## Get file

After uploading the file, you can verify the API has successfully received the files by calling `files.get`.

It lets you get the file metadata that have been uploaded to the File API that are associated with the Cloud project your API key belongs to. Only the `name` (and by extension, the `uri`) are unique. Only use the `displayName` to identify files if you manage uniqueness yourself.

In [18]:
file = client.files.get(name=sample_file.name)
print(f"Retrieved file '{file.name}' as: {sample_file.uri}")

Retrieved file 'files/zyg60km4wauf' as: https://generativelanguage.googleapis.com/v1beta/files/zyg60km4wauf


## Generate content

After uploading the file, you can make `interactions.create` requests that reference the File API URI.

In [20]:
MODEL_ID = "gemini-3.5-flash"  # @param ["gemini-3.1-flash-lite", "gemini-2.5-flash-lite", "gemini-3.5-flash", "gemini-2.5-flash", "gemini-2.5-pro", "gemini-3-flash-preview", "gemini-2.5-flash-preview", "gemini-3.1-pro-preview"] {"allow-input":true, isTemplate: true}

interaction = client.interactions.create(
    model=MODEL_ID,
    input=[
        {"type": "text", "text": "Describe the image with a creative description."},
        {"type": "document", "uri": sample_file.uri},
    ],
)

print(interaction.steps[-1].content[0].text)

In the humble margins of a blue-lined notebook, a revolution in student transit is taking shape. This hand-drawn blueprint, rendered in confident strokes of ballpoint blue ink, reveals the "Jetpack Backpack"—the ultimate "sleeper" accessory for the modern commuter.

At first glance, it poses as a standard, lightweight rucksack, complete with comfortable padded straps and enough room to swallow a massive 18-inch laptop. But look closer at the base, where the mundane meets the miraculous. A pair of retractable boosters are deployed, exhaling whimsical, swirling plumes of steam. This isn’t just a bag; it’s a clean, green flying machine designed to turn a ten-minute trek across campus into a thirty-second aerial sprint.

The sketch captures the delightful tension between futuristic ambition and practical reality. While it boasts high-tech features like USB-C charging and eco-friendly propulsion, the inventor remains grounded, noting an endearingly honest fifteen-minute battery life—just en

## Multiple files

You'll often want to upload multiple files at once. Here's a quick example how you can do it:

In [22]:
!git clone -q --depth 1 https://github.com/googleapis/python-genai

In [23]:
import pathlib

files = []
for p in pathlib.Path("python-genai").rglob('*.py'):
  if 'test' in str(p):
    continue
  f = client.files.upload(file=p, config={'display_name': str(p)})
  # The API doesn't see the file name, so add those to the list of parts.
  files.append(f"<<<File: {str(p)}>>>")
  files.append(f)
  print('.', end='')

....................................................................................................................................................................................


In [24]:
interaction = client.interactions.create(
    model=MODEL_ID,
    input=[
        {"type": "text", "text": "Hi, could you give me a summary of this code base?"},
    ] + [{"type": "document", "uri": f.uri} for f in files],
)

print(interaction.steps[-1].content[0].text)

AttributeError: 'str' object has no attribute 'uri'

## Use files from Google Cloud Storage

The Gemini API supports accessing objects stored in Google Cloud Storage (GCS). As GCS objects have a different security model to the Gemini API, you first need to register files before you can use them.

To register a GCS object for use in the File API, you need to authenticate with an identity that has **Storage Object Viewer** permissions, and with the appropriate OAuth scope enabled. The mechanism varies depending on the environment, for example you can [download service account credentials](https://docs.cloud.google.com/iam/docs/keys-create-delete) to use on your own infrastructure, or run on a Compute Engine instance that has [a configured service account](https://docs.cloud.google.com/compute/docs/access/create-enable-service-accounts-for-instances).

For this notebook, you will configure your user credentials via `gcloud`.

In [26]:
# Use the same project ID you're using for the Gemini API project
PROJECT_ID = "your-project-id"  # @param {type: "string"}

!gcloud config set project {PROJECT_ID}
!gcloud auth application-default login --no-launch-browser --scopes="https://www.googleapis.com/auth/cloud-platform,https://www.googleapis.com/auth/devstorage.read_only"

SyntaxError: unterminated string literal (detected at line 5) (File_API.ipynb:cell_25, line 5)

Now retrieve your credentials and register your object(s).

Note: If you are registering files separately from your generation code, you will need to create a `genai.Client` object initialised with an API key. The API key is unused by this method, but is still required to ensure the client is not connected to the Vertex AI backend.

Upload files to a bucket in the project you're using: https://docs.cloud.google.com/storage/docs/uploading-objects. Then get the URIs and register the files:

In [28]:
import google.auth

credentials, project_id = google.auth.default()

registered_gcs_files = client.files.register_files(
    auth=credentials,
    uris=["gs://your-bucket/some-file.pdf"]
)

RefreshError: Certificate config or certificate file not found after multiple retries. Token binding protection is failing. You can turn off this protection by setting GOOGLE_API_PREVENT_AGENT_TOKEN_SHARING_FOR_GCP

The `register_files` endpoint returns the files associated with the supplied URIs. You can then use these directly in your prompt with `generate_content`, or store the `name`s to use at a later time.

In [30]:
interaction = client.interactions.create(
    model=MODEL_ID,
    input=[
        {"type": "text", "text": "What are these documents about?"},
    ] + [{"type": "document", "uri": f.uri} for f in registered_gcs_files.files],
)

print(interaction.steps[-1].content[0].text)

NameError: name 'registered_gcs_files' is not defined

## Delete files

Files are automatically deleted after 2 days or you can manually delete them using `files.delete()`.

In [32]:
client.files.delete(name=sample_file.name)
print(f"Deleted {sample_file.name}.")

Deleted files/zyg60km4wauf.


## Supported text types

As well as supporting media uploads, the File API can be used to embed text files, such as Python code, or Markdown files, into your prompts.

This example shows you how to load a markdown file into a prompt using the File API.

In [34]:
# Download a markdown file and ask a question.
from IPython.display import Markdown, display
import urllib.request

urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/google-gemini/cookbook/main/CONTRIBUTING.md",
    "contrib.md"
)

md_file = client.files.upload(
    file="contrib.md",
    config={
        "display_name": "CONTRIBUTING.md",
    },
)

interaction = client.interactions.create(
    model=MODEL_ID,
    input=[
        {"type": "document", "uri": md_file.uri},
        {"type": "text", "text": "Summarize this document in a few sentences."},
    ],
)
display(Markdown(interaction.steps[-1].content[0].text))

<IPython.core.display.Markdown object>


Some common text formats are automatically detected, such as `text/x-python`, `text/html` and `text/markdown`. If you are using a file that you know is text, but is not automatically detected by the API as such, you can specify the MIME type as `text/plain` explicitly.

In [36]:
# Download some C++ code and force the MIME as text when uploading.
import urllib.request
from IPython.display import Markdown, display

urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/google/gemma.cpp/main/examples/hello_world/run.cc",
    "gemma.cpp"
)

cpp_file = client.files.upload(
    file="gemma.cpp",
    config={
        "display_name":"gemma.cpp",
        "mime_type":"text/plain"
    }
)

interaction = client.interactions.create(
    model=MODEL_ID,
    input=[
        {"type": "document", "uri": cpp_file.uri},
        {"type": "text", "text": "Explain this code and identify any potential bugs."},
    ],
)
display(Markdown(interaction.steps[-1].content[0].text))

<IPython.core.display.Markdown object>


## External files and URLs

In addition to the File API, you can provide some URLs to the Gemini API directly, including public HTTPS URLs and signed URLs of content on cloud storage platforms.

### Direct HTTPS URLs

As long as the URLs are public, not paywalled and contain a supported file type, you can add them to your prompt and the API will download the content while processing the prompt.

In [39]:
MODEL_ID = "gemini-3.5-flash"  # @param ["gemini-3.1-flash-lite", "gemini-2.5-flash-lite", "gemini-3.5-flash", "gemini-2.5-flash", "gemini-2.5-pro", "gemini-3-flash-preview", "gemini-2.5-flash-preview", "gemini-3.1-pro-preview"] {"allow-input":true, isTemplate: true}

interaction = client.interactions.create(
    model=MODEL_ID,
    input=[
        {"type": "text", "text": "Describe the image with a creative description."},
        {"type": "image", "uri": "https://storage.googleapis.com/generativeai-downloads/images/jetpack.jpg"},
    ],
)

print(interaction.steps[-1].content[0].text)

RateLimitError: Error code: 429 - {'error': {'message': 'Resource has been exhausted (e.g. check quota).', 'code': 'too_many_requests'}}

### Signed URLs

If your data is stored in a private cloud storage system, such as S3 or Azure Blob Storage, you can generate a signed URL and pass it to the Gemini API to use directly, saving you from downloading the file from your own storage and managing its lifecycle on another platform.

The URLs are requested each time they are used, so revoking access on the host platform will ensure the content is not available through an uncached Gemini API request.

The following code demonstrates how to sign an S3 object. See your specific provider's documentation to learn how to sign your specific objects.

```python
# pip install boto3

import boto3

s3 = boto3.client('s3')
signed_url = s3.generate_presigned_url(
    'get_object',
    Params={
      # Set your bucket name here.
      'Bucket': 'my-bucket-name',
      # And the path to the object here.
      'Key': 'document.pdf'
    },
    ExpiresIn=3600  # In seconds
)
```

In [41]:
MODEL_ID = "gemini-3.5-flash"  # @param ["gemini-3.1-flash-lite", "gemini-2.5-flash-lite", "gemini-3.5-flash", "gemini-2.5-flash", "gemini-2.5-pro", "gemini-3-flash-preview", "gemini-2.5-flash-preview", "gemini-3.1-pro-preview"] {"allow-input":true, isTemplate: true}

# Obtain the signed URL from your provider, or see the S3
# example code above.
signed_url = "https://..."  # @param {type: "string"}

interaction = client.interactions.create(
    model=MODEL_ID,
    input=[
        {"type": "text", "text": "Summarise this document."},
        {"type": "document", "uri": signed_url},
    ],
)

print(interaction.steps[-1].content[0].text)

BadRequestError: Error code: 400 - {'error': {'message': 'Cannot fetch content from the provided URL.', 'code': 'invalid_request'}}

## Next Steps
### Useful API references:

For more information about the File API, check its [API reference](https://ai.google.dev/api/files). You will also find more code samples [in this folder](../quickstarts/file-api).

### Related examples

Check those examples using the File API to give you more ideas on how to use that very useful feature:
* Share [Voice memos](../examples/Voice_memos.ipynb) with Gemini API and brainstorm ideas
* Analyze videos to [classify](../examples/Analyze_a_Video_Classification.ipynb) or [summarize](../examples/Analyze_a_Video_Summarization.ipynb) them

### Continue your discovery of the Gemini API

If you're not already familiar with it, learn how [tokens are counted](../quickstarts/Counting_Tokens.ipynb). Then check how to use the File API to use [Audio](../quickstarts/Audio.ipynb) or [Video](../quickstarts/Video_understanding.ipynb) files with the Gemini API.
